<a href="https://colab.research.google.com/github/mohanraju-07/LLM/blob/main/medical_QA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset


In [ ]:
dataset = load_dataset("medalpaca/medical_meadow_medqa",split="train")
dataset = dataset.shuffle(seed=42)

train_test = dataset.train_test_split(test_size=0.2)
val_test = train_test["test"].train_test_split(test_size=0.5)

train_data = train_test["train"]
val_data = val_test["train"]
test_data = val_test["test"]

print(f"Train : {len(train_data)}")
print(f"Val   : {len(val_data)}")
print(f"Test  : {len(test_data)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Train : 8142
Val   : 1018
Test  : 1018


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
train_data[0]

{'input': "Q:A 39-year-old woman presents to the family medicine clinic to be evaluated by her physician for weight gain. She reports feeling fatigued most of the day despite eating a healthy diet and exercising regularly. The patient smokes a half-pack of cigarettes daily and has done so for the last 23 years. She is employed as a phlebotomist by the Red Cross. She has a history of hyperlipidemia for which she takes atorvastatin. She is unaware of her vaccination history, and there is no documented record of her receiving any vaccinations. Her heart rate is 76/min, respiratory rate is 14/min, temperature is 37.3°C (99.1°F), body mass index (BMI) is 33 kg/m2, and blood pressure is 128/78 mm Hg. The patient appears alert and oriented. Lung and heart auscultation are without audible abnormalities. The physician orders a thyroid panel to determine if that patient has hypothyroidism. Which of the following recommendations may be appropriate for the patient at this time?? \n{'A': 'Hepatitis

In [ ]:
print(type(tokenizer))
print(tokenizer("hello", truncation=True))

<class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>
{'input_ids': [21820, 1], 'attention_mask': [1, 1]}


In [ ]:
def tokenize(batch):
  model_inputs =tokenizer(
      batch['input'],
      truncation=True
  )
  labels = tokenizer(
      batch['output'],
      truncation=True
  )
  model_inputs["labels"] = labels["input_ids"]
  return model_inputs

tokenized_train = train_data.map(tokenize, batched=True)
tokenized_test = test_data.map(tokenize, batched=True)
tokenized_val = val_data.map(tokenize, batched=True)


Map:   0%|          | 0/8142 [00:00<?, ? examples/s]

Map:   0%|          | 0/1018 [00:00<?, ? examples/s]

Map:   0%|          | 0/1018 [00:00<?, ? examples/s]

In [ ]:
from transformers import DataCollatorForSeq2Seq
from torch.utils.data import DataLoader


data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer,padding=True)

train_dataloader = DataLoader(
    tokenized_train,
    shuffle=True,
    batch_size=8,
    collate_fn = data_collator
)

val_dataloader = DataLoader(
    tokenized_val,
    batch_size=8,
    collate_fn = data_collator,
)




In [ ]:
from transformers import get_scheduler
import torch


optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

num_epochs=3
num_training_steps = num_epochs * len(train_dataloader)

lr_scheduler = get_scheduler(
    name="linear",
    num_training_steps = num_training_steps,
    optimizer=optimizer,
    num_warmup_steps=0
)

print(f"Total training steps: {num_training_steps}")


Total training steps: 3054


In [ ]:
!pip install evaluate rouge_score -q

In [ ]:
import evaluate

metric = evaluate.load("rouge")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Using device: {device}")

Using device: cuda


In [ ]:
print(tokenized_train.column_names)

['input', 'instruction', 'output', 'input_ids', 'attention_mask', 'labels']


In [ ]:
text_columns = ["input", "output", "instruction"]

tokenized_train = tokenized_train.remove_columns(text_columns)
tokenized_val   = tokenized_val.remove_columns(text_columns)
tokenized_test  = tokenized_test.remove_columns(text_columns)

tokenized_train.set_format("torch")
tokenized_val.set_format("torch")
tokenized_test.set_format("torch")

print(tokenized_train.column_names)

['input_ids', 'attention_mask', 'labels']


In [ ]:
print(tokenized_train.column_names)
print(tokenized_train[0])

['input_ids', 'attention_mask', 'labels']
{'input_ids': tensor([ 1593,    10,   188,  6352,    18,  1201,    18,  1490,  2335,  6621,
           12,     8,   384,  4404,  5998,    12,    36, 14434,    57,   160,
        10027,    21,  1293,  2485,     5,   451,  2279,  1829, 13034,    26,
          167,    13,     8,   239,     3,  3565,  3182,     3,     9,  1695,
         3178,    11, 22668,  3842,     5,    37,  1868,  7269,     7,     3,
            9,   985,    18,  5745,    13, 22893,  1444,    11,    65,   612,
           78,    21,     8,   336,  1902,   203,     5,   451,    19,  8152,
           38,     3,     9,     3,   102,   107,   109,  4045,    32,    51,
          343,    57,     8,  1624,  4737,     5,   451,    65,     3,     9,
          892,    13,  6676, 19437, 11658,    21,    84,   255,  1217,     3,
         1016,   900,  8547,    77,     5,   451,    19, 24111,    13,   160,
        24639,   892,     6,    11,   132,    19,   150, 15552,  1368,    13,
        

In [ ]:
print(tokenized_train.column_names)
print(tokenized_val.column_names)

['input_ids', 'attention_mask', 'labels']
['input_ids', 'attention_mask', 'labels']


In [ ]:
print(type(data_collator))

<class 'transformers.data.data_collator.DataCollatorForSeq2Seq'>


In [ ]:
import torch

# Clear GPU memory first
torch.cuda.empty_cache()

# Reduce batch size
train_dataloader = DataLoader(tokenized_train, batch_size=2, shuffle=True,  collate_fn=data_collator)
val_dataloader   = DataLoader(tokenized_val,   batch_size=2, shuffle=False, collate_fn=data_collator)

In [ ]:
import matplotlib.pyplot as plt

train_losses = []
val_losses   = []

accumulation_steps = 4

for epoch in range(num_epochs):
    model.train()
    total_training_loss = 0
    optimizer.zero_grad()

    for i, batch in enumerate(train_dataloader):
        batch   = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss    = outputs.loss / accumulation_steps
        loss.backward()

        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

        total_training_loss += loss.item()
    avg_train_loss = total_training_loss / len(train_dataloader)
    train_losses.append(avg_train_loss)



In [ ]:
model.eval()
total_val_loss  = 0
all_predictions = []
all_references  = []

for batch in val_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
        outputs   = model(**batch)
        generated = model.generate(batch["input_ids"], max_new_tokens=64)

    total_val_loss += outputs.loss.item()
    decoded_preds  = tokenizer.batch_decode(generated,       skip_special_tokens=True)
    decoded_preds  = tokenizer.batch_decode(generated, skip_special_tokens=True)

    labels = batch["labels"]
    labels[labels == -100] = tokenizer.pad_token_id
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    all_predictions.extend(decoded_preds)
    all_references.extend(decoded_labels)

avg_val_loss = total_val_loss / len(val_dataloader)
rouge_result = metric.compute(predictions=all_predictions, references=all_references)

print(f"Val Loss  : {avg_val_loss:.4f}")
print(f"ROUGE-1   : {rouge_result['rouge1']:.4f}")
print(f"ROUGE-2   : {rouge_result['rouge2']:.4f}")
print(f"ROUGE-L   : {rouge_result['rougeL']:.4f}")

Val Loss  : 3.0909
ROUGE-1   : 0.2846
ROUGE-2   : 0.1827
ROUGE-L   : 0.2828


In [ ]:
print(
    f"Epoch {epoch + 1} | "
    f"Train Loss: {avg_train_loss:.4f} | "
    f"Val Loss: {avg_val_loss:.4f} | "
    f"ROUGE-1: {rouge_result['rouge1']:.4f} | "
    f"ROUGE-2: {rouge_result['rouge2']:.4f} | "
    f"ROUGE-L: {rouge_result['rougeL']:.4f}"
)

print("✅ Training and evaluation complete for this epoch!")

Epoch 3 | Train Loss: 0.9459 | Val Loss: 3.0909 | ROUGE-1: 0.2846 | ROUGE-2: 0.1827 | ROUGE-L: 0.2828
✅ Training and evaluation complete for this epoch!
